# `neutralize` — Cross-Sectional Risk Neutralization

This is the heart of the framework and the step most often done wrong. **On each day** we run **one OLS regression across the ~30 stocks** (a *cross-sectional* regression — not a time-series one) of that day's standardized signal on the stocks' risk exposures. The **residual** is the part of the signal *not* explained by beta, size or sector — i.e. the idiosyncratic alpha.

$$ \text{signal}_{i,t} = \gamma_{0,t} + \gamma_{\beta,t}\,\beta_{i,t} + \gamma_{s,t}\,\text{size}_{i,t} + \sum_k \gamma_{k,t}\,\text{sector}_{i,k} + \underbrace{\varepsilon_{i,t}}_{\text{neutralized signal}} $$

### The big picture
* **Preprocess each day** by winsorizing at ±3σ (clip crazy outliers) *then* z-scoring (put every day on a common scale). Order matters — clip first so outliers don't distort the z-score.
* **Solve with `np.linalg.lstsq`**, not a matrix inverse: the sector dummies can be collinear and would break a plain inverse; least-squares handles that gracefully.
* **Why beta, not book-to-market?** Fundamentals are reported late and get restated, leaking future info into a backtest. A price-based beta is available in real time and look-ahead-free.
* An optional **PCA mode** neutralizes against statistical factors instead of named ones.

The specifics of each tricky line are in the `#` comments below.

In [ ]:
%run config.ipynb

In [ ]:
"""Cross-sectional preprocessing + daily residualization (the neutralization engine)."""
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from typing import Dict


def winsorize_and_standardize_cross_section(row: np.ndarray) -> np.ndarray:
    """Clip one day's cross-section at +/-3 sigma, then z-score it. Missing names stay NaN."""
    row = np.asarray(row, dtype=float)
    out = row.copy()        # copy first so stocks with no value today keep their NaN
    mask = ~np.isnan(row)   # which stocks actually have a value today
    if mask.sum() < 2:      # need at least two names before "spread across stocks" means anything
        return out

    valid = row[mask]
    mu, sd = valid.mean(), valid.std()
    if sd == 0:             # every stock identical -> the day carries no cross-sectional info
        out[mask] = 0.0
        return out

    # WHY WINSORIZE BEFORE STANDARDIZING (the order really matters)?
    # Both the mean and the standard deviation are NON-robust statistics: one extreme value
    # drags the mean and inflates the std. If we z-scored first, that inflated std would divide
    # every stock, squashing the genuine spread among the other 29 names into a narrow band
    # around zero. One blow-up would effectively erase the information in the rest of the
    # cross-section. So we clip the tails first, then compute mean/std on the tamed data.
    lo, hi = mu - 3.0 * sd, mu + 3.0 * sd
    clipped = np.clip(valid, lo, hi)

    # Now z-score, recomputing mean and std AFTER clipping so the scale reflects the tamed data.
    # Standardizing each day separately is what makes days comparable to each other: the raw
    # dispersion of a signal changes a lot between calm and volatile markets, and without this
    # step a wild day would silently carry more weight than a quiet one in everything downstream.
    # (np.std uses ddof=0, the population formula. For putting a day on a common scale the n
    # vs n-1 choice is just a constant rescaling of the whole day, so it does not matter here.)
    cmu, csd = clipped.mean(), clipped.std()
    out[mask] = 0.0 if csd == 0 else (clipped - cmu) / csd
    return out


def neutralize_signal(raw_signal: pd.DataFrame, returns: pd.DataFrame,
                      exposures: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Residualize the signal against risk factors with ONE OLS per day (cross-sectional)."""
    cols = raw_signal.columns
    neutralized = pd.DataFrame(np.nan, index=raw_signal.index, columns=cols)
    common_index = raw_signal.index.intersection(returns.index)

    # Line the exposures up to exactly the same stock order as the signal, once, up front. Then
    # inside the daily loop we can index everything positionally and trust that column j means
    # the same stock in every array -- misalignment here would silently pair one stock's signal
    # with another stock's beta, and nothing would ever error.
    beta_df = exposures['beta'].reindex(columns=cols)
    size_df = exposures['size'].reindex(columns=cols)
    sector_mat = exposures['sectors'].reindex(cols).values.astype(float)  # [N stocks x K sectors]
    n_sec = sector_mat.shape[1]

    if config.NEUTRALIZATION_MODE == 'explicit':
        # How many stocks do we need before a fit is meaningful? With k regressors and only k
        # observations, OLS passes exactly through every point: all residuals are 0 and there is
        # no signal left to measure. The useful quantity is the RESIDUAL DEGREES OF FREEDOM,
        # n - k, and we insist on at least a couple of them before trusting a day.
        n_reg = 2 + n_sec + 1          # beta + size + K sector dummies + intercept
        min_obs = n_reg + 2
        sectors_ok = np.all(np.isfinite(sector_mat), axis=1)  # stock has a usable sector row

        # ---- the daily cross-sectional loop: ONE regression per trading day ----
        # This loop is over DAYS, not stocks, and that is the whole point of the method: on each
        # date we fit a fresh regression across the ~30 stocks alive that day. Because the fit is
        # re-estimated daily, the risk-factor coefficients are free to change over time -- we
        # never assume one fixed relationship for the whole 8-year sample.
        for date in common_index:
            # Today's raw signal, standardized across stocks -> this is the y we try to explain.
            sig = winsorize_and_standardize_cross_section(raw_signal.loc[date].values)
            beta = beta_df.loc[date].values
            size = size_df.loc[date].values

            # A stock only enters today's regression if it has ALL of signal, beta, size and
            # sector. Dropping incomplete rows (rather than filling them) keeps the fit honest:
            # an imputed regressor would invent a relationship that the data never showed.
            valid = (~np.isnan(sig)) & (~np.isnan(beta)) & (~np.isnan(size)) & sectors_ok
            if valid.sum() < min_obs:
                continue

            # Design matrix X = [intercept, beta, size, sector dummies].
            X = np.column_stack([np.ones(valid.sum()), beta[valid], size[valid], sector_mat[valid]])
            y = sig[valid]

            # Solve for the coefficients that minimize the sum of squared residuals.
            # We use lstsq instead of the textbook formula coef = inv(X'X) @ X'y because X'X can
            # easily be singular here -- e.g. on a day when every Energy name is missing, that
            # sector's dummy column is all zeros. inv() would raise or return garbage; lstsq goes
            # through an SVD and returns a sensible minimum-norm solution instead.
            coef, *_ = np.linalg.lstsq(X, y, rcond=None)

            # *** THE KEY LINE: the residual IS the neutralized signal. ***
            # OLS does not merely "reduce" the relationship with the regressors, it removes it
            # exactly. The normal equations that define the OLS solution say X'(y - X@coef) = 0,
            # i.e. the residual vector is ORTHOGONAL to every column of X. So on this day the
            # leftover has, by construction, exactly zero sample correlation with beta, with
            # size, and with each sector dummy.
            # That is what "neutralized" means and why it matters: whatever predictive power
            # survives in this leftover CANNOT be explained away as "you were just buying
            # high-beta names" or "you were just long Tech". It is idiosyncratic, stock-specific
            # alpha -- which is exactly the thing we are trying to isolate and measure.
            neutralized.loc[date, cols[valid]] = y - X @ coef

    elif config.NEUTRALIZATION_MODE == 'pca':
        # Alternative flavour: instead of neutralizing against factors we NAMED (beta, size,
        # sector), let the data tell us what the common factors are. PCA on the recent return
        # covariance extracts the directions that explain the most co-movement; we strip those
        # out. Useful when you do not trust your sector labels, at the cost of interpretability
        # -- the components are statistical artefacts with no economic name attached.
        k = config.PCA_COMPONENTS
        for date in common_index:
            t_idx = returns.index.get_loc(date)
            if t_idx < config.BETA_LOOKBACK:            # need a full trailing window first
                continue
            # Strictly PAST returns only (iloc stops at t_idx, excluding today) -- fitting the
            # PCA on data that included today would leak information into the signal.
            hist = returns.iloc[t_idx - config.BETA_LOOKBACK:t_idx].dropna(axis=1, how='any')
            assets = hist.columns
            if len(assets) < k + 2:
                continue
            # Each stock's loadings on the top-k statistical factors become the regressors.
            comps = PCA(n_components=k).fit(hist[assets]).components_.T  # [len(assets) x k]
            sig = winsorize_and_standardize_cross_section(raw_signal.loc[date, assets].values)
            valid = ~np.isnan(sig)
            if valid.sum() < k + 2:
                continue
            X = np.column_stack([np.ones(valid.sum()), comps[valid]])
            y = sig[valid]
            coef, *_ = np.linalg.lstsq(X, y, rcond=None)
            neutralized.loc[date, assets[valid]] = y - X @ coef
    else:
        raise ValueError(f"Unknown NEUTRALIZATION_MODE: {config.NEUTRALIZATION_MODE!r}")

    # One final winsorize + z-score pass over the residuals. Regression residuals come out with
    # whatever scale the fit happened to leave behind, and that scale drifts from day to day.
    # Re-standardizing puts the neutralized factor back on the SAME footing as the raw signal,
    # so the raw-vs-neutralized comparison later is apples-to-apples rather than a comparison of
    # two differently-scaled quantities.
    return neutralized.apply(winsorize_and_standardize_cross_section, axis=1, result_type='broadcast')


print("neutralization helpers ready: winsorize_and_standardize_cross_section(), neutralize_signal()")